<a href="https://colab.research.google.com/github/MusaR10/AAI2025/blob/2026fall/Excercise_2_Prompt_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2: Code Generation with ReAct Prompting

**Goal:** Generate Python code using reasoning before execution.

**ReAct cycle used in this notebook:**
1. **Reason**: Gemini restates the requirements and lists edge cases
2. **Plan**: Gemini writes a step-by-step plan
3. **Generate code**: Gemini writes the function
4. **Run**: the notebook executes the code in Colab
5. **Observe**: the notebook runs 20 tests and records what passed and failed
6. **Fix**: the observation is sent back to Gemini, which explains each failure and returns corrected code

Steps 4–6 repeat until all tests pass (maximum 4 iterations).

**Coding task:** `parse_duration(text)` converts strings like `"1h 30m"` into seconds, with strict input validation.

**Tools:** Google Colab, Python 3, Google Gemini API (`gemini-3.1-flash-lite`), pandas

## Setup
Connect to Gemini using the API key stored in Colab Secrets.

In [2]:
import google.generativeai as genai
from google.colab import userdata, files
from IPython.display import display
from PIL import Image as PILImage
import time
import os
import json
import re
import signal
import traceback
import pandas as pd

# Connect to Gemini
genai.configure(api_key=userdata.get("Prompt_Engineering_Key"))

model = genai.GenerativeModel("gemini-3.1-flash-lite")

print("Gemini initialized successfully.")

Gemini initialized successfully.


## Helper functions
`ask()` sends a prompt to Gemini (with a retry if the rate limit is hit), and `show()` prints each ReAct stage with a clear label.

In [4]:
def ask(prompt, temperature=0.2, retries=3):
    """Send one prompt to Gemini and return the text reply."""
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt, generation_config={"temperature": temperature})
            time.sleep(3)  # small pause to stay under free-tier rate limits
            return response.text.strip()
        except Exception as e:
            print(f"  (attempt {attempt + 1} failed: {str(e)[:100]} ... retrying)")
            time.sleep(20)
    raise RuntimeError("Gemini call failed after several retries.")


def show(label, data):
    """Print one stage of the ReAct cycle with a label."""
    print(f"\n--- {label} ---")
    print(data)

## Read Gemini's response
Gemini is told to answer with labeled sections (`### THOUGHT`, `### PLAN`, `### CODE`). This function splits the reply into those sections and pulls out the Python code block so it can be run.

In [5]:
def parse_response(text):
    """Split the reply into its labeled sections and extract the Python code block."""
    sections = {}
    pattern = r"###\s*([A-Z ]+?)\s*\n(.*?)(?=\n###\s*[A-Z ]+?\s*\n|\Z)"
    for match in re.finditer(pattern, text, re.S):
        sections[match.group(1).strip()] = match.group(2).strip()

    code_match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    code = code_match.group(1).strip() if code_match else ""
    return sections, code

## Task specification
The exact requirements for the function: inputs, outputs, error handling, examples, and allowed libraries. This is inserted into every ReAct prompt so Gemini always works from the same rules.

In [6]:
SPEC = """Write a Python function:

    def parse_duration(text: str) -> int

It converts a duration string into a total number of SECONDS.

INPUT FORMAT
- A string made of one or more parts. Each part is a whole number followed immediately by a unit letter.
- Units: d = days (86400 s), h = hours (3600 s), m = minutes (60 s), s = seconds.
- Units are case-insensitive ("1H" is the same as "1h").
- Spaces between parts are optional ("1h30m" and "1h 30m" are both valid). Leading/trailing spaces are ignored.
- Units must appear in the order d, h, m, s, and each unit can appear at most once.
- Zero values are allowed ("0s" returns 0).

OUTPUT
- Return the total number of seconds as an int.

ERROR HANDLING
- If text is not a str, raise TypeError.
- Raise ValueError with a helpful message for: an empty or whitespace-only string, a number without a unit ("5", "1h30"), a unit without a number ("h"), an unknown unit ("5x"), a repeated unit ("1h 2h"), units out of order ("30m 1h"), negative numbers ("-5m"), or decimals ("1.5h").

EXAMPLES
parse_duration("1h 30m")  -> 5400
parse_duration("2d4h")    -> 187200
parse_duration(" 45S ")   -> 45
parse_duration("30m 1h")  -> raises ValueError
parse_duration(90)        -> raises TypeError

CONSTRAINTS
- Use only the Python standard library `re` module. No other imports.
- Include type hints and a short docstring.
- The function must not print anything or ask for input."""

print("Specification loaded.")

Specification loaded.


## ReAct Stages 1–3: Reason → Plan → Generate code
The first prompt. Gemini must reason about the requirements and edge cases (**THOUGHT**), plan the logic (**PLAN**), and only then write the code (**CODE**). It is told up front that its code will be run and that it will get test results back.

In [7]:
def react_first_attempt():
    prompt = f"""ROLE: You are a careful senior Python developer. You solve coding tasks with the ReAct method: you REASON before you ACT, and you use test results to fix your code.

HOW THIS WORKS (ReAct cycle):
1. THOUGHT: reason about the requirements and edge cases.
2. PLAN: plan the logic step by step.
3. CODE: act by writing the code.
4. Your code will then be RUN against a test suite, and you will receive an OBSERVATION with the results.
5. If any tests fail, you will be asked to FIX the code using that observation. This repeats until all tests pass.

TASK SPECIFICATION:
{SPEC}

RESPONSE FORMAT (use these exact headings, in this order, and nothing else):
### THOUGHT
Restate the requirements in your own words and list every edge case you must handle.
### PLAN
A numbered, step-by-step plan for the function's logic.
### CODE
Exactly one ```python code block containing only the import(s) and the function definition. No example usage, no tests, no print statements."""
    return ask(prompt, temperature=0.2)

## Test cases
Tests used in the Observe stage: 8 valid inputs with their expected number of seconds, and 12 invalid inputs that must raise the correct error.

In [8]:
TEST_CASES = [
    # (input, expected result OR expected error type)
    ("1h 30m", 5400),
    ("2d4h", 187200),
    (" 45S ", 45),
    ("1H 5M 10S", 3910),
    ("0s", 0),
    ("1d 0h 0m 1s", 86401),
    ("10h", 36000),
    ("1d2h3m4s", 93784),
    ("", ValueError),
    ("   ", ValueError),
    ("5", ValueError),
    ("1h30", ValueError),
    ("h", ValueError),
    ("5x", ValueError),
    ("1h 2h", ValueError),
    ("30m 1h", ValueError),
    ("-5m", ValueError),
    ("1.5h", ValueError),
    (90, TypeError),
    (None, TypeError),
]

print(f"{len(TEST_CASES)} test cases loaded.")

20 test cases loaded.


## ReAct Stage 4: Run the code
Executes Gemini's code in Colab. Before running, it checks for forbidden imports or functions so only safe code is executed. It returns the loaded function, or an error message if the code could not run.

In [10]:
FORBIDDEN = ["import os", "import sys", "subprocess", "shutil", "socket", "requests",
             "open(", "eval(", "exec(", "__import__", "input("]


def run_code(code):
    """Load the generated code. Returns (function, None) or (None, error_message)."""
    if not code:
        return None, "No ```python code block was found in the response."

    blocked = [item for item in FORBIDDEN if item in code]
    if blocked:
        return None, f"The code was NOT run because it uses forbidden items: {blocked}. Only the `re` module is allowed."

    namespace = {}
    try:
        exec(code, namespace)
    except Exception:
        return None, "The code failed to load:\n" + traceback.format_exc(limit=2)

    func = namespace.get("parse_duration")
    if not callable(func):
        return None, "No function named parse_duration was defined."
    return func, None

## ReAct Stage 5: Observe the results
Runs every test on the loaded function (with a 2-second time limit per test) and writes an **observation**: how many tests passed, plus the exact input, expected result, and actual result for each failure. This text becomes the feedback for the Fix stage.

In [11]:
def _timeout_handler(signum, frame):
    raise TimeoutError("test took longer than 2 seconds")

signal.signal(signal.SIGALRM, _timeout_handler)


def is_error(expected):
    return isinstance(expected, type) and issubclass(expected, Exception)


def observe(func, run_error):
    """Run all tests and return (passed, total, observation_text)."""
    total = len(TEST_CASES)
    if run_error:
        return 0, total, run_error

    passed, failures = 0, []
    for test_input, expected in TEST_CASES:
        call = f"parse_duration({test_input!r})"
        try:
            signal.alarm(2)
            result = func(test_input)
            signal.alarm(0)
            if is_error(expected):
                failures.append(f"{call}: expected {expected.__name__}, but it returned {result!r}")
            elif result == expected and type(result) is int:
                passed += 1
            else:
                failures.append(f"{call}: expected {expected!r} (int), but got {result!r} ({type(result).__name__})")
        except Exception as e:
            signal.alarm(0)
            if is_error(expected) and isinstance(e, expected):
                passed += 1
            else:
                wanted = expected.__name__ if is_error(expected) else repr(expected)
                failures.append(f"{call}: expected {wanted}, but it raised {type(e).__name__}: {e}")

    observation = f"{passed}/{total} tests passed."
    if failures:
        observation += "\nFailing tests:\n" + "\n".join("- " + f for f in failures)
    return passed, total, observation

## ReAct Stage 6: Fix using the observation
The fix prompt. Gemini receives its previous code and the observation, must explain the root cause of each failure (**THOUGHT**), plan the changes (**FIX PLAN**), and return the complete corrected function (**CODE**).

In [12]:
def react_fix(previous_code, observation, iteration):
    prompt = f"""ROLE: You are a careful senior Python developer using the ReAct method (Reason -> Act -> Observe -> Fix).

TASK SPECIFICATION:
{SPEC}

This is iteration {iteration}. Your previous code was run against the test suite.

YOUR PREVIOUS CODE:
```python
{previous_code}
```

OBSERVATION (test results):
{observation}

RESPONSE FORMAT (use these exact headings, in this order, and nothing else):
### THOUGHT
For each failing test, explain the root cause in your previous code.
### FIX PLAN
List the specific changes you will make, and explain why they will not break the tests that already pass.
### CODE
Exactly one ```python code block with the complete corrected function (not just the changed lines). Same constraints as before: only the `re` import, type hints, a docstring, and no print statements."""
    return ask(prompt, temperature=0.2)

## The ReAct loop
Connects all six stages. Iteration 1 uses the first-attempt prompt; every later iteration uses the fix prompt with the latest observation. The loop stops as soon as all tests pass, or after 4 iterations.

In [13]:
def react_loop(max_iterations=4):
    history = []
    code, observation = None, None

    for iteration in range(1, max_iterations + 1):
        print("\n" + "=" * 75)
        print(f"ITERATION {iteration}" + ("  (first attempt)" if iteration == 1 else "  (fix attempt)"))
        print("=" * 75)

        # Stages 1-3 (first attempt) or Stage 6 (fix)
        if iteration == 1:
            response = react_first_attempt()
        else:
            response = react_fix(code, observation, iteration)
        sections, code = parse_response(response)

        show("REASON (THOUGHT)", sections.get("THOUGHT", "(no THOUGHT section returned)"))
        plan_key = "PLAN" if iteration == 1 else "FIX PLAN"
        show(plan_key, sections.get(plan_key, sections.get("PLAN", "(no plan section returned)")))
        show("GENERATED CODE", code or "(no code returned)")

        # Stage 4: run
        func, run_error = run_code(code)

        # Stage 5: observe
        passed, total, observation = observe(func, run_error)
        show("RUN + OBSERVE (test results)", observation)

        history.append({"Iteration": iteration, "Tests passed": f"{passed}/{total}",
                        "All passed": passed == total, "code": code})

        if passed == total:
            print(f"\n✅ All {total} tests passed on iteration {iteration}. Stopping the loop.")
            break
        print(f"\n🔁 {total - passed} test(s) failed. Sending the observation back to Gemini to fix.")
    else:
        print(f"\n⚠️ Stopped after {max_iterations} iterations without passing every test.")

    return history

In [14]:
history = react_loop() #Run the loop


ITERATION 1  (first attempt)

--- REASON (THOUGHT) ---
The goal is to parse a duration string into total seconds based on units (d, h, m, s).
Requirements:
- Units: d=86400, h=3600, m=60, s=1.
- Case-insensitive, optional spaces.
- Strict order: d -> h -> m -> s.
- Each unit at most once.
- Errors: TypeError for non-string, ValueError for empty/whitespace, missing units, missing numbers, unknown units, repeated units, out-of-order units, negative numbers, or decimals.

Edge cases:
- "0s" -> 0.
- "1d 1h 1m 1s" -> valid.
- "1h 30m" (out of order) -> invalid.
- "1.5h" -> invalid.
- "-5m" -> invalid.
- "1h30" -> invalid (missing unit for 30).
- "h" -> invalid (missing number).

--- PLAN ---
1. Validate input type is `str`.
2. Strip whitespace and check if empty; raise `ValueError` if so.
3. Define a regex pattern to capture parts: `(\d+)([dhms])`.
4. Use `re.findall` to extract all parts.
5. Validate that the entire string is consumed by the regex (i.e., no invalid characters or malformed

In [15]:
#Summary
display(pd.DataFrame(history)[["Iteration", "Tests passed", "All passed"]])

final_code = history[-1]["code"]
print("\nFINAL CODE:\n")
print(final_code)

,Iteration,Tests passed,All passed
0,1,20/20,True



FINAL CODE:

import re

def parse_duration(text: str) -> int:
    """
    Converts a duration string (e.g., '1d 2h') into total seconds.
    """
    if not isinstance(text, str):
        raise TypeError("Input must be a string.")
    
    text = text.strip()
    if not text:
        raise ValueError("Input string is empty or whitespace-only.")

    # Regex to find all valid parts
    pattern = re.compile(r'(\d+)([dhms])', re.IGNORECASE)
    matches = pattern.findall(text)
    
    # Check if the string contains anything invalid (e.g., "1h30", "1.5h", "5x")
    # Remove all valid parts and spaces; if anything remains, it's an error.
    cleaned = re.sub(r'\s+', '', text)
    for val, unit in matches:
        cleaned = cleaned.replace(f"{val}{unit}", "", 1)
    
    if cleaned:
        raise ValueError(f"Invalid format or unknown characters: {cleaned}")

    multipliers = {'d': 86400, 'h': 3600, 'm': 60, 's': 1}
    order = ['d', 'h', 'm', 's']
    
    total_seconds = 0
    last_idx = 

In [16]:
namespace = {}
exec(final_code, namespace)
parse_duration = namespace["parse_duration"]

print("Valid inputs:")
for text in ["1h 30m", "2d4h", " 45S ", "1d2h3m4s", "0s"]:
    print(f"  parse_duration({text!r:14}) = {parse_duration(text):>7} seconds")

print("\nInvalid inputs (error handling):")
for text in ["30m 1h", "1.5h", "5x", "", 90]:
    try:
        parse_duration(text)
        print(f"  parse_duration({text!r:14}) -> no error raised")
    except (ValueError, TypeError) as e:
        print(f"  parse_duration({text!r:14}) -> {type(e).__name__}: {e}")

Valid inputs:
  parse_duration('1h 30m'      ) =    5400 seconds
  parse_duration('2d4h'        ) =  187200 seconds
  parse_duration(' 45S '       ) =      45 seconds
  parse_duration('1d2h3m4s'    ) =   93784 seconds
  parse_duration('0s'          ) =       0 seconds

Invalid inputs (error handling):
  parse_duration('30m 1h'      ) -> ValueError: Units must be in order (d, h, m, s) and appear only once.
  parse_duration('1.5h'        ) -> ValueError: Invalid format or unknown characters: 1.
  parse_duration('5x'          ) -> ValueError: Invalid format or unknown characters: 5x
  parse_duration(''            ) -> ValueError: Input string is empty or whitespace-only.
  parse_duration(90            ) -> TypeError: Input must be a string.


## Before: plain prompt (no ReAct)
The original version of this exercise used a one-line prompt with no specification, no reasoning stage, and no feedback loop. It is run here against the same 20 tests for comparison.

In [17]:
def plain_prompt():
    prompt = """Write a Python function called parse_duration that converts a duration string like "1h 30m" into seconds."""
    return ask(prompt, temperature=0.2)


baseline_response = plain_prompt()
_, baseline_code = parse_response(baseline_response)
baseline_func, baseline_error = run_code(baseline_code)
baseline_passed, total, baseline_observation = observe(baseline_func, baseline_error)

show("BEFORE: CODE FROM PLAIN PROMPT", baseline_code or "(no code block returned)")
show("BEFORE: TEST RESULTS", baseline_observation)

ERROR:tornado.access:503 POST /v1beta/models/gemini-3.1-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1092.59ms


5400
2700
7210
30

--- BEFORE: CODE FROM PLAIN PROMPT ---
import re

def parse_duration(duration_str):
    """
    Converts a duration string (e.g., '1h 30m', '45m', '2h 10s') into total seconds.
    """
    # Define the multipliers for each unit
    units = {
        'h': 3600,
        'm': 60,
        's': 1
    }
    
    # Regex to find numbers followed by a unit (h, m, or s)
    pattern = r'(\d+)\s*([hms])'
    matches = re.findall(pattern, duration_str.lower())
    
    total_seconds = 0
    for value, unit in matches:
        total_seconds += int(value) * units.get(unit, 0)
        
    return total_seconds

# Examples:
print(parse_duration("1h 30m"))    # Output: 5400
print(parse_duration("45m"))       # Output: 2700
print(parse_duration("2h 10s"))    # Output: 7210
print(parse_duration("30s"))       # Output: 30

--- BEFORE: TEST RESULTS ---
5/20 tests passed.
Failing tests:
- parse_duration('2d4h'): expected 187200 (int), but got 14400 (int)
- parse_duration('1d 0h 0m 1s'): e

## Before vs. after comparison
Test results for the plain prompt, the first ReAct attempt, and the final ReAct result.

In [18]:
comparison = pd.DataFrame([
    {"Version": "Before: plain one-line prompt", "Tests passed": f"{baseline_passed}/{total}"},
    {"Version": "After: ReAct, iteration 1", "Tests passed": history[0]["Tests passed"]},
    {"Version": f"After: ReAct, final (iteration {len(history)})", "Tests passed": history[-1]["Tests passed"]},
])
display(comparison)

,Version,Tests passed
0,Before: plain one-line prompt,5/20
1,"After: ReAct, iteration 1",20/20
2,"After: ReAct, final (iteration 1)",20/20
